# Lesson 03 · Multilayer networks & the XOR problem: one hidden layer does what one neuron can't

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/03-mlp-va-xor-en.ipynb)

This notebook accompanies [Lesson 03 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/03-mlp-va-xor.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
Lesson 01 ended with a failure: no perceptron can solve XOR. Here we write, in numpy, a network with a
**hidden layer** ourselves, see how it transforms the four XOR points, let it learn on its own, and then look at the
hidden layer of a real MNIST network.

Section 0 sets out all of the lesson's theory first, without running anything; the sections after it rebuild each
idea in numpy and check it with numbers.

**What you'll do:**
1. Prove that no straight line solves XOR, and see the perceptron rule cycle forever on it.
2. Choose the weights of a 2 → 2 → 1 network yourself so that it solves XOR, and look at the four points in the
   **hidden space**.
3. Let the network learn its weights from 100 different initializations, and see when it gets stuck.
4. See that more hidden neurons draw more complex regions, and that depth multiplies the number of regions faster
   than width.
5. Train the course's 784 → 128 → 64 → 10 MNIST network, then measure how much its hidden layer helps the
   perceptron of Lesson 01.

**Background:** [Lesson 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/perceptron/) (the
learning rule, $z = xW + b$) and [Lesson 02 · Activation functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-kich-hoat/)
(sigmoid, ReLU, why a nonlinear $f$ is needed); partial derivatives and the Taylor expansion (section 0.5 uses them).
**Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

Every number in the notebook is computed by the cells below, and matches the numbers in the reel and the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 0 · Theory before code

This section gathers all of the lesson's theory in one place and doesn't need any cell to run. Sections 3 to 8
rebuild each idea in numpy, first on the four XOR points and then on MNIST; the table at the end of this section
says where each idea is checked. Notation as in Lesson 01 and MLSysBook: $x$ is a row vector, a layer computes
$xW + b$ and then applies the activation function $f$ elementwise, $H$ is the number of hidden neurons, and the step
function of Lesson 01 is $f(z) = 1$ for $z \ge 0$, $f(z) = 0$ for $z < 0$.

### 0.1 · One neuron can't solve XOR

Two sets of points are **linearly separable** if some straight line (with more inputs: a hyperplane) puts every
point of one set on one side and every point of the other set on the other side. A neuron
$z = x_1 w_1 + x_2 w_2 + b$, $\hat y = f(z)$ predicts 1 on the half-plane $z \ge 0$ and 0 on the other half, so some
$(w_1, w_2, b)$ gets every point right if and only if the two labels are linearly separable.

XOR (“exclusive or”) has four points: $(0,0)$ and $(1,1)$ with label 0, $(0,1)$ and $(1,0)$ with label 1. A neuron
that gets all four right must satisfy

$$b < 0, \qquad w_2 + b \ge 0, \qquad w_1 + b \ge 0, \qquad w_1 + w_2 + b < 0.$$

Adding the two middle inequalities gives $w_1 + w_2 + 2b \ge 0$, that is, $w_1 + w_2 + b \ge -b > 0$, which
contradicts the last one. So no $(w_1, w_2, b)$ gets all four points right: XOR is not linearly separable.

Geometry adds one more fact. Both sides of a line, $\{z \ge 0\}$ and $\{z < 0\}$, are **convex**: if a side contains
two points, it contains the whole segment between them. The segment joining the two points labeled 1 and the segment
joining the two points labeled 0 are the two diagonals of the square, which cross at the center
$(\tfrac12, \tfrac12)$. If one side contained both points of one label and the other side both points of the other
label, the center would lie on both sides, which is impossible. The argument doesn't care which label is on which
side, so it also rules out the opposite case, getting all four points wrong (that is, solving XNOR). So every line
gets at least one and at most three of the four points right.

XOR is the two-input case of the **parity** function. Minsky and Papert (1969) analyzed limits of this kind in
depth, for a single learnable layer placed on top of fixed features. Networks of several threshold neurons, on the
other hand, had been known since McCulloch and Pitts (1943) to compute any logical function, XOR included;
section 0.4 builds one.

### 0.2 · The perceptron rule on XOR: it never stops, it cycles

Rosenblatt's rule from Lesson 01 takes a misclassified point $(x, y)$ and updates
$w \leftarrow w + \eta\,(y - \hat y)\,x$, $b \leftarrow b + \eta\,(y - \hat y)$; it stops only when no point is
misclassified. Lesson 01's convergence theorem (Novikoff, 1962) needs linearly separable data, so it doesn't apply
here. By section 0.1, every $(w, b)$ leaves at least one point misclassified, so the rule **never stops**.

It doesn't drift off to infinity either. Block and Levin (1970) proved that when the data is not separable, the
perceptron's weights stay **bounded**. With XOR, every $x_i$ is 0 or 1 and $y - \hat y = \pm 1$, so each update adds
$\pm\eta$ to $b$ and to the $w_i$ with $x_i = 1$. If $(w_1^0, w_2^0, b^0)$ are the starting weights, then
$(w_1, w_2, b)$ always lies on the lattice $(w_1^0, w_2^0, b^0) + \eta\,\mathbb Z^3$. A bounded set contains only
finitely many points of a lattice. The algorithm's state is $(w, b)$ together with the point to be examined next, so
it too takes only finitely many values, and the next state is a fixed function of the current one. Such a sequence
must return to an earlier state, and from there it repeats exactly as before, forever: the algorithm falls into a
**cycle**. The argument uses exact arithmetic; section 3 runs both the floating-point version and an integer-only
version.

### 0.3 · The hidden layer: a change of space

A $2 \to H \to 1$ network puts $H$ neurons side by side, as at the end of Lesson 01, one column of $W_1$ per neuron,
and lets an output neuron read their results:

$$h = f(xW_1 + b_1), \qquad \hat y = f(hW_2 + b_2).$$

With $x$ of size $1 \times 2$, $W_1$ is $2 \times H$, $b_1$ and $h$ are $1 \times H$, $W_2$ is $H \times 1$, and
$b_2$ is a single number. The network has $2H + H + H + 1 = 4H + 1$ parameters, and each point costs $2H + H = 3H$
multiply–accumulates (MACs). $h$ is the **hidden layer**: nobody gives it labels, and it isn't in the data. Stack the
points as the rows of a matrix $P$ and the whole set goes through the hidden layer in one matrix product,
$f(PW_1 + b_1)$, with $b_1$ added to every row.

There are two ways to read the same network. In the input space $(x_1, x_2)$, each hidden neuron is still a
perceptron, that is, a straight line. The second reading matters more: the hidden layer is a **change of
representation** $x \mapsto h$ that gives each point new coordinates in the **hidden space** $(h_1, \dots, h_H)$,
and the output neuron is just a Lesson 01 perceptron placed on $h$. So, once the hidden layer is fixed, some
$W_2, b_2$ gets every point right if and only if the points $h$, each carrying the label of the $x$ it came from, are
linearly separable in the hidden space. A **multilayer perceptron** (MLP) stacks several such changes:
$h_\ell = f(h_{\ell - 1} W_\ell + b_\ell)$, with $h_0 = x$.

What happens if you drop $f$ is a 🤔 question in section 5, so section 0 leaves it for there.

### 0.4 · A hand-set solution: OR and AND

Take $f$ to be the step function, $H = 2$, and

$$W_1 = \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}, \qquad b_1 = (-0.5, -1.5), \qquad
W_2 = \begin{bmatrix} 1 \\ -2 \end{bmatrix}, \qquad b_2 = -0.5.$$

On 0/1 inputs:

- $h_1 = f(x_1 + x_2 - 0.5) = 1$ if and only if at least one input is 1: $h_1 = x_1 \lor x_2$ (OR).
- $h_2 = f(x_1 + x_2 - 1.5) = 1$ if and only if both inputs are 1: $h_2 = x_1 \land x_2$ (AND).
- $\hat y = f(h_1 - 2h_2 - 0.5)$. If $h_2 = 0$, then $\hat y = f(h_1 - 0.5) = h_1$. If $h_2 = 1$, then
  $h_1 - 2 - 0.5 < 0$, so $\hat y = 0$. Hence $\hat y = h_1 \land \lnot h_2$: “OR but not AND”.

That is exactly XOR: OR is wrong only at $(1,1)$, and “not AND” removes exactly that point. The network has
$4 \cdot 2 + 1 = 9$ parameters and costs $3 \cdot 2 = 6$ MACs per point. Each hidden neuron is a perceptron that
solves a separable problem (OR, AND); what one neuron can't do, three neurons arranged in two layers can.

### 0.5 · Learning the weights: sigmoid, squared error, gradient descent

Nine parameters can be set by hand; the MNIST network's more than a hundred thousand (section 0.8) can't, so they
have to be **learned**. The perceptron rule doesn't work for the hidden layer: the hidden layer has no labels, so
there is no way to know what the “right” $h$ is. The step function gives no direction either: its derivative is 0
at every $z \ne 0$, so nudging a weight slightly either leaves $\hat y$ unchanged or makes it jump between 0 and 1
(Lesson 02). Section 6 takes the modern approach:

1. Replace the step with the **sigmoid** in both layers, $h = \sigma(xW_1 + b_1)$ and $\hat y = \sigma(hW_2 + b_2)$,
   with $\sigma(z) = 1/(1 + e^{-z})$. The function $\sigma$ is smooth and strictly increasing, gives
   $\hat y \in (0, 1)$, and $\sigma(z) \ge \tfrac12 \iff z \ge 0$. A network counts as **solving XOR** when the
   threshold $\hat y \ge \tfrac12$ is right at all four points; that threshold is exactly the line $hW_2 + b_2 = 0$
   in the hidden space, so the output is still a perceptron on $h$, as in section 0.3.
2. Collect all the parameters in a vector $\theta$ of $4H + 1$ numbers, and measure the error with the **mean squared
   error** over the four points: $\mathcal L(\theta) = \tfrac14 \sum_{i=1}^{4} (\hat y_i - y_i)^2$.
3. **Gradient descent** (Lesson 05): repeat $\theta \leftarrow \theta - \eta\, \nabla_\theta \mathcal L$, that is,
   nudge every parameter against the gradient, the direction in which $\mathcal L$ grows fastest around the current
   point.

The derivative is computed by **central differences**, with no formula needed: with $e_k$ the $k$-th unit vector,

$$\frac{\partial \mathcal L}{\partial \theta_k} \approx
\frac{\mathcal L(\theta + \varepsilon e_k) - \mathcal L(\theta - \varepsilon e_k)}{2\varepsilon}.$$

A Taylor expansion says how accurate this is. Writing $\partial_k$ for $\partial / \partial \theta_k$:

$$\mathcal L(\theta \pm \varepsilon e_k) = \mathcal L \pm \varepsilon\, \partial_k \mathcal L
+ \tfrac{\varepsilon^2}{2}\, \partial_k^2 \mathcal L \pm \tfrac{\varepsilon^3}{6}\, \partial_k^3 \mathcal L
+ O(\varepsilon^4).$$

Subtracting the two lines cancels the even-order terms, so

$$\frac{\mathcal L(\theta + \varepsilon e_k) - \mathcal L(\theta - \varepsilon e_k)}{2\varepsilon}
= \partial_k \mathcal L + \frac{\varepsilon^2}{6}\, \partial_k^3 \mathcal L + O(\varepsilon^4).$$

The error is proportional to $\varepsilon^2$: halving $\varepsilon$ divides the error by about 4. The price is $2p$
evaluations of $\mathcal L$ per step, where $p = 4H + 1$ is the number of parameters; for the MNIST network of
section 0.8, $2p$ is hundreds of thousands of forward passes. Backpropagation (Lesson 06) computes exactly that
$\nabla_\theta \mathcal L$ at a cost of about one forward pass and one backward pass; section 8 uses it as a black
box.

### 0.6 · More hidden neurons: decision regions

**One hidden neuron is never enough.** With $H = 1$, $W_1$ is a single column $w$ of size $2 \times 1$; write
$b_1 = c$, $W_2 = a$, $b_2 = d$. Then $h = \sigma(xw + c)$ is a single number, and
$\hat y = \sigma(ah + d) \ge \tfrac12 \iff ah + d \ge 0$. If $a > 0$, that is equivalent to $h \ge -d/a$; since
$\sigma$ is strictly increasing, that means $xw + c \ge t$ for some threshold $t$ (or always true, or never true). If
$a < 0$, it is equivalent to $xw + c \le t$. If $a = 0$, $\hat y$ doesn't depend on $x$. In every case the region
$\hat y \ge \tfrac12$ is a half-plane, the whole plane, or empty, so by section 0.1 no $2 \to 1 \to 1$ network can
represent XOR.

**More neurons, more half-planes.** With the step function, hidden neuron $j$ equals 1 on a half-plane $S_j$, and
$\hat y$ depends only on which of the $S_j$ contain $x$. So the region predicted 1 is a union of some of the pieces
into which the $H$ lines cut the plane. Choose $W_2 = (1, \dots, 1)^\top$ and $b_2 = -(H - \tfrac12)$: then
$\sum_j h_j - (H - \tfrac12) \ge 0$ only when all $H$ neurons equal 1, so the output is an AND (like $h_2$ in
section 0.4), and the region predicted 1 is the intersection $S_1 \cap \dots \cap S_H$, a convex region.

**A polygon around a circle.** For $H \ge 3$, take $u_j = \big(\cos\tfrac{2\pi j}{H},\ \sin\tfrac{2\pi j}{H}\big)$. The neuron
$h_j = f(1 - x u_j^\top)$ equals 1 on the half-plane that contains the center, bounded by the tangent to the unit
circle at $u_j$. The intersection of these $H$ half-planes is the regular $H$-sided polygon circumscribed about the
circle, each of its sides at distance exactly 1 from the center. Joining the center to the vertices cuts the polygon
into $H$ isosceles triangles, each with angle $2\pi/H$ at the center, height 1 and base $2\tan(\pi/H)$, so the
polygon has area $H\tan(\pi/H)$. The disk lies entirely inside the polygon, so the region where the network answers
wrong has area

$$A(H) = H\tan\frac{\pi}{H} - \pi = \frac{\pi^3}{3H^2} + O\!\left(H^{-4}\right),$$

where the second equality comes from $\tan t = t + t^3/3 + O(t^5)$ with $t = \pi/H$. For large $H$, doubling the
number of neurons shrinks the wrong region about 4 times.

### 0.7 · Universal approximation, and deep or wide

**The universal approximation theorem** (Cybenko, 1989, for the sigmoid; Hornik, 1991; Leshno et al., 1993). Let
$f$ be continuous and not a polynomial (for a continuous $f$, Leshno et al. proved that this condition is necessary
and sufficient). For every continuous function $g$ on a closed and bounded domain $K \subset \mathbb R^n$ and every
$\delta > 0$, there exist a number $H$ and a network with one hidden layer of $H$ neurons,
$F(x) = f(xW_1 + b_1)\,W_2 + b_2$, such that $|F(x) - g(x)| < \delta$ for every $x \in K$.

The theorem is an **existence** statement: it says such a network exists, but it doesn't say how large $H$ must be,
and it doesn't exhibit the weights. The polygon of section 0.6 is the theorem's intuition, not the theorem itself:
the function approximated there (1 inside the circle, 0 outside) is not continuous, and the error is measured as an
area rather than at every point.

**Wide: each neuron adds one kink.** With a single input $x \in \mathbb R$ and ReLU, a hidden layer of $m$ neurons
computes

$$F(x) = \sum_{j=1}^{m} a_j\, \mathrm{ReLU}(w_j x + c_j) + d.$$

Each term is a piecewise linear function that bends (changes slope) at most at one point, $x = -c_j / w_j$, and a sum
can bend only where one of its terms bends. So $F$ has at most $m$ kinks, that is, at most $m + 1$ linear pieces. On
$[0, 1]$ it has exactly $m + 1$ pieces when the kinks are distinct, all lie in $(0, 1)$, and every $a_j$ and $w_j$ is
nonzero, because then each kink changes the slope by $a_j |w_j| \ne 0$.

**Deep: each layer doubles the number of pieces.** The “roof” function
$g(x) = 2\,\mathrm{ReLU}(x) - 4\,\mathrm{ReLU}(x - \tfrac12)$ equals $2x$ on $[0, \tfrac12]$ and $2 - 2x$ on
$[\tfrac12, 1]$: a layer of 2 neurons that maps $[0, 1]$ onto itself, going up and then down. Write
$g^{(k)} = g \circ \dots \circ g$ ($k$ times); it is a network with $k$ hidden layers, each made of the two neurons
$\mathrm{ReLU}(u)$ and $\mathrm{ReLU}(u - \tfrac12)$ of the number $u$ computed by the layer before. Induction on $k$:
$g^{(1)} = g$ consists of 2 pieces, each going linearly once all the way between 0 and 1, with slopes of opposite
signs. Suppose $g^{(k)}$ consists of $2^k$ such pieces, with slopes of alternating signs. On each of those pieces,
$g^{(k+1)} = g \circ g^{(k)}$ goes through both halves of $g$, so it splits into two pieces with slopes of opposite
signs, each again going all the way between 0 and 1, and the slopes still alternate in sign. So $g^{(k)}$ has exactly
$2^k$ pieces with $2k$ neurons, while a single hidden layer needs $m + 1 \ge 2^k$, that is, at least $2^k - 1$
neurons. **Depth multiplies the number of pieces; width only adds to it.** Telgarsky (2015) proved a stronger
result: there are functions that deep networks compute, while shallow networks need a number of neurons that grows
exponentially to approximate them.

### 0.8 · The course's MLP: 784 → 128 → 64 → 10

MLSysBook's MNIST network has two ReLU hidden layers:

$$h_1 = \mathrm{ReLU}(xW_1 + b_1), \qquad h_2 = \mathrm{ReLU}(h_1 W_2 + b_2), \qquad z = h_2 W_3 + b_3,$$

with $x$ of size $1 \times 784$ (a 28×28 image flattened into a row, brightness divided by 255), $W_1$ of size
$784 \times 128$, $W_2$ of size $128 \times 64$ and $W_3$ of size $64 \times 10$. $z$ holds 10 scores, one per
digit, and the network picks $\arg\max_j z_j$; Lesson 04 turns those 10 scores into probabilities with softmax. A
layer $n \to n'$ costs $nn'$ MACs and has $nn' + n'$ parameters ($W$ and $b$), so each image costs

$$784 \cdot 128 + 128 \cdot 64 + 64 \cdot 10 = 100{,}352 + 8{,}192 + 640 = 109{,}184 \text{ MACs},$$

and the network has $109{,}184 + 128 + 64 + 10 = 109{,}386$ parameters. The first layer, connected to all 784
pixels, accounts for almost all of the cost. The network learns by gradient descent on mini-batches of images, with
softmax and cross-entropy (Lesson 04) instead of the sigmoid and the squared error, and with the gradient computed by
backpropagation (Lesson 06).

### Where each piece of theory is checked

| Theory | Checked with numbers in |
|---|---|
| 0.1 XOR is not separable; every line gets one to three points right | section 3 (the random lines) |
| 0.2 the perceptron rule never stops and falls into a cycle | section 3 (the long run, floating-point and integer versions) |
| 0.3 the hidden layer, $4H + 1$ parameters, $3H$ MACs; the output is a perceptron on $h$ | sections 4 and 5 |
| 0.4 the OR/AND solution | section 4 |
| 0.5 sigmoid, $\mathcal L$, gradient descent; the central-difference error is proportional to $\varepsilon^2$ | section 6 |
| 0.6 one hidden neuron is not enough; the polygon, $A(H) = H\tan(\pi/H) - \pi$ | section 7 |
| 0.7 one layer: at most $m + 1$ pieces; $k$ layers: $2^k$ pieces (universal approximation: intuition only, section 7) | section 7 |
| 0.8 784 → 128 → 64 → 10: MACs and parameters | section 8 |

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import itertools
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, ScalarFormatter

# The course's palette (colorblind-safe), always paired with shapes: label 0 = orange circle, label 1 = blue square
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # 0 → ½ → 1
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 109,184 · 0.5 · −0.13 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every linear axis (colorbars included) in the English format; call right before
    plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(tick))


print(num(109184, 0), num(0.5, 1), num(-0.1263, 2), num(-2.8e-17, 1), pct(0.9727))

## 2 · The data: the four XOR points (and MNIST for section 8)

XOR (“exclusive or”) outputs 1 when its two inputs are **different**. The whole dataset has only four points. We
stack them as the rows of a matrix $P$, as with every dataset in the course: each row is a row vector
$x = (x_1, x_2)$.

In [ ]:
P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)  # row i: the input (x₁, x₂) of point i
Y = np.array([0, 1, 1, 0])  # labels: 1 when x₁ ≠ x₂
for i, (p, y) in enumerate(zip(P, Y)):
    print(f"point {i}: x = ({p[0]:.0f}, {p[1]:.0f})  →  y = {y}")


def show_points(ax, X=P, y=Y, s=170, legend=True):
    """Draw the points: label 0 = orange circle, label 1 = blue square (color always paired with shape)."""
    for (a, b), lab in zip(X, y):
        ax.scatter(a, b, s=s, marker="s" if lab else "o", color=BLUE if lab else ORANGE, ec="white", lw=2, zorder=3)
    if legend:
        ax.scatter([], [], s=80, marker="o", color=ORANGE, label="y = 0")
        ax.scatter([], [], s=80, marker="s", color=BLUE, label="y = 1")
        ax.legend(frameon=False, loc="upper right", fontsize=9)


fig, ax = plt.subplots(figsize=(3.6, 3.6))
show_points(ax)
for (a, b), lab in zip(P, Y):
    ax.annotate(f"({a:.0f}, {b:.0f})", (a, b), xytext=(0, -20), textcoords="offset points", ha="center", fontsize=9)
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x₁", ylabel="x₂", title="XOR: four points, two labels")
ax.set_aspect("equal")
fmt_axes(plt.gcf())
plt.show()

The two orange points lie on one diagonal, the two blue points on the other.

Section 8 uses MNIST (70,000 28×28 images of handwritten digits), downloaded exactly as in Lesson 01, with its MD5
hashes checked.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images / 255.0, labels.astype(np.int64)  # each row is one image: 784 numbers in [0, 1]


fetch()
Xtr, ytr = load("train")
Xte, yte = load("t10k")
print("training:", Xtr.shape, " test:", Xte.shape)

## 3 · Back to Lesson 01: one neuron loses to XOR

The theory of this section is in sections 0.1 and 0.2. We rerun exactly the experiment at the end of Lesson 01,
which is also the experiment in the reel and the demo: the perceptron starts from $w = (0.3, -0.2)$, $b = 0.1$, with
learning rate $\eta = 0.1$. Each step updates the first
misclassified point after the one just updated.

The function `perceptron()` below can be reused for any small set of points (we'll need it in section 5). The
function `z_of()` adds in exactly the order the reel and the demo use, $x_1 w_1 + x_2 w_2 + b$: a few updates bring
$z$ very close to 0, and then floating-point rounding error decides the sign of $z$.

In [ ]:
def step(z):
    return (np.asarray(z) >= 0).astype(int)


def z_of(X, w, b):
    """z = x₁w₁ + x₂w₂ + … + b for every row of X."""
    z = X[:, 0] * w[0]
    for i in range(1, X.shape[1]):
        z = z + X[:, i] * w[i]
    return z + b


def perceptron(X, y, w, b, eta=0.1, max_updates=16):
    """Rosenblatt's rule (Lesson 01). Returns the states (w, b, number of points correct, cursor) after each update."""
    w, n, cursor = np.array(w, float), len(y), 0
    states = [(w.copy(), b, int((step(z_of(X, w, b)) == y).sum()), cursor)]
    for _ in range(max_updates):
        wrong = step(z_of(X, w, b)) != y
        if not wrong.any():  # all correct: stop
            break
        j = next((cursor + q) % n for q in range(n) if wrong[(cursor + q) % n])
        d = y[j] - step(z_of(X[j : j + 1], w, b))[0]  # +1 or −1
        w, b = w + eta * d * X[j], b + eta * d
        cursor = (j + 1) % n
        states.append((w.copy(), b, int((step(z_of(X, w, b)) == y).sum()), cursor))
    return states


run = perceptron(P, Y, [0.3, -0.2], 0.1)
history = [s[2] for s in run]
print("points classified correctly after each update:", history)
assert history == [1, 1, 2, 2, 3, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2], "differs from the reel and the demo: check again"

Exactly the sequence in the reel and the demo: never more than 3/4.

### What if it runs longer?
The reel stops after only 16 updates. The cell below runs 1,000 updates and checks whether the algorithm's state
(made of $w$, $b$ and the point to be examined next) ever repeats.

In [ ]:
def find_cycle(states):
    """(t₀, period): the first update t₀ from which the whole state (w, b, cursor) repeats exactly."""
    seen = {}
    for t, (w, b, _, cursor) in enumerate(states):
        key = (*w, b, cursor)
        if key in seen:
            return seen[key], t - seen[key]
        seen[key] = t
    return None


long_run = perceptron(P, Y, [0.3, -0.2], 0.1, max_updates=1000)
t0, period = find_cycle(long_run)
print(f"{num(len(long_run) - 1, 0)} updates; at most {max(s[2] for s in long_run)}/4 points correct")
print(f"from update {t0} on, the state repeats exactly every {period} updates:")
for t in range(t0, t0 + period):
    w_t, b_t, n_ok, _ = long_run[t]
    print(f"  after update {t:>2}: w = ({num(w_t[0], 1)}, {num(w_t[1], 1)}), b = {num(b_t, 1)}  →  {n_ok}/4 correct")
assert max(s[2] for s in long_run) < 4 and (t0, period) == (11, 4)

# The same algorithm in exact arithmetic: multiply everything by 10 (w = (3, −2), b = 1, η = 1), only integers remain
exact = perceptron(P, Y, [3, -2], 1, eta=1, max_updates=1000)
print("integers, first 16 updates:", [s[2] for s in exact[:17]])
print(f"integers: at most {max(s[2] for s in exact)}/4 correct; cycle (t₀, length) = {find_cycle(exact)}")
z_bad = z_of(P[3:], *long_run[3][:2])[0]
print("z of the point (1, 1) after update 3, computed in floating point:", f"{z_bad:.1e}".replace("-", "−"))
assert max(s[2] for s in exact) < 4

Never all 4 right. More than that, from update 11 on the algorithm enters a **cycle**: four states repeat exactly,
forever, and $w$ neither grows nor shrinks to 0. This is the “cycling” theorem of Block & Levin (1970): when the
data is not separable, the perceptron's weights always stay bounded. Here $x$ consists only of 0s and 1s, so each
update adds or subtracts exactly $0.1$ to a few weights; bounded weights then allow only finitely many states, and
the algorithm is forced to return to an earlier one (the full argument is in section 0.2).

The last three lines of the cell above show a small detail: in exact arithmetic (everything multiplied by 10 so that
only integers remain), the sequence of points correct differs from the reel's right after update 3. After update 3,
the point $(1, 1)$ should have $z$ exactly 0, but floating point computes a tiny negative number, so the sign of $z$
follows the rounding error. The conclusion doesn't change: both cycle without ever getting 4/4.

### Why no straight line can do it
This repeats the proof of section 0.1. A neuron predicts 1 when $z = x_1 w_1 + x_2 w_2 + b \ge 0$. To get all four
XOR points right, it needs all of these at once:

$$\underbrace{b < 0}_{(0,0)\to 0}, \qquad \underbrace{w_2 + b \ge 0}_{(0,1)\to 1}, \qquad
\underbrace{w_1 + b \ge 0}_{(1,0)\to 1}, \qquad \underbrace{w_1 + w_2 + b < 0}_{(1,1)\to 0}.$$

Adding the two middle inequalities gives $w_1 + w_2 + 2b \ge 0$, that is, $w_1 + w_2 + b \ge -b > 0$, which
contradicts the last inequality. No $(w_1, w_2, b)$ satisfies all four. Let's check with 100,000 random lines:

In [ ]:
lines = np.random.default_rng(0).normal(size=(100_000, 3))  # each row: (w₁, w₂, b) of one line
n_correct = (step(P @ lines[:, :2].T + lines[:, 2]) == Y[:, None]).sum(axis=0)
print(f"{num(len(lines), 0)} random lines:")
for k, count in enumerate(np.bincount(n_correct, minlength=5)):
    print(f"{k}/4 points correct: {num(count, 0):>6} lines")
assert n_correct.max() == 3 and n_correct.min() == 1

No line gets 4/4, and no line gets all 4 wrong either (getting all of them wrong would mean solving the opposite
problem, XNOR, which is not separable either). The “best” line gets only 3/4.

### Minsky & Papert (1969)
The book *Perceptrons* by Marvin Minsky and Seymour Papert studies exactly this: what can a single learnable layer,
placed on top of fixed features, compute? XOR is the two-input case of the **parity** function (whether the number of
inputs equal to 1 is odd or even). Their theorem says that a perceptron can compute the parity of $n$ pixels only if
at least one feature sees all $n$ pixels. The book does not say that multilayer networks can't solve XOR: it had been
known since McCulloch & Pitts (1943) that networks of threshold neurons can compute any logical function. What was
missing was a way to **learn** the weights of the hidden layers, and Minsky & Papert were skeptical that such a way
existed. The story that “one book killed neural networks” is an oversimplification, but the book did help stall this
line of research, until backpropagation (Rumelhart, Hinton & Williams, 1986;
[Lesson 06](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/)) became popular.

## 4 · Add a hidden layer: two neurons draw two lines

One line isn't enough, so use two (sections 0.3 and 0.4). Put two neurons side by side, as at the end of Lesson 01,
one column of $W_1$ per neuron:

$$h = f(x W_1 + b_1), \qquad \hat y = f(h W_2 + b_2)$$

Here $x$ is a $1 \times 2$ row vector and $W_1$ is $2 \times 2$, so $h = (h_1, h_2)$ has two numbers: that is the
**hidden layer**, because nobody sees it or gives it labels. $W_2$, of size $2 \times 1$, connects $h$ to the output
neuron. We choose the weights ourselves, with $f$ the step function from Lesson 01:

- $h_1 = f(x_1 + x_2 - 0.5)$: equals 1 when **at least one** input is 1 (OR).
- $h_2 = f(x_1 + x_2 - 1.5)$: equals 1 when **both** inputs are 1 (AND).
- $\hat y = f(h_1 - 2h_2 - 0.5)$: “OR but not AND”, which is exactly XOR.

In [ ]:
W1 = np.array([[1.0, 1.0], [1.0, 1.0]])  # column 1: neuron h₁ (OR), column 2: neuron h₂ (AND)
b1 = np.array([-0.5, -1.5])
W2 = np.array([[1.0], [-2.0]])  # 2 × 1
b2 = np.array([-0.5])

H_hand = step(P @ W1 + b1)  # the hidden layer of all 4 points at once: Z = PW₁ + b₁, of size 4 × 2
y_hand = step(H_hand @ W2 + b2)[:, 0]
for p, h, yh, y in zip(P, H_hand, y_hand, Y):
    print(f"x = ({p[0]:.0f}, {p[1]:.0f})  →  h = ({h[0]}, {h[1]})  →  ŷ = {yh}   (label {y})")
assert (y_hand == Y).all()
n_params = W1.size + b1.size + W2.size + b2.size
print(
    f"{(y_hand == Y).sum()}/4 correct with {n_params} parameters; MACs per point: {W1.size} + {W2.size} = {W1.size + W2.size}"
)

All four points right, with 9 parameters. Each hidden neuron is still just a perceptron, that is, a straight line
$z = 0$ in the $(x_1, x_2)$ plane. The whole network combines those two lines.

## 5 · The hidden space: XOR becomes separable

In the terms of section 0.3, $h$ gives each point new coordinates; this section looks at the four XOR points in those
coordinates.

### 🤔 Predict first
The hidden layer turns each point $x$ into a new point $h = (h_1, h_2)$. In the $(h_1, h_2)$ plane, where do the
four XOR points lie: still at the four corners of the square, or somewhere else? And how many updates does the
perceptron of Lesson 01, starting from exactly $w = (0.3, -0.2)$, $b = 0.1$, need on these points $h$?

In [ ]:
hidden_run = perceptron(H_hand.astype(float), Y, [0.3, -0.2], 0.1)
w_h, b_h, n_ok, _ = hidden_run[-1]
print("the 4 points in the hidden space:", [tuple(int(v) for v in h) for h in H_hand])
print(
    f"perceptron on h: {len(hidden_run) - 1} updates → {n_ok}/4 correct  (w = ({num(w_h[0], 1)}, {num(w_h[1], 1)}), b = {num(b_h, 1)})"
)
assert n_ok == 4 and len(hidden_run) - 1 == 2

The hidden layer **merges** the two points $(0,1)$ and $(1,0)$ into the same point $h = (1, 0)$. The four corners of
the square become just three points, and a straight line separates them: only 2 updates are needed, from exactly the
starting point that failed in section 3.

The figure below puts the two spaces side by side. On the left is the input space $(x_1, x_2)$: the dashed line is
the line of $h_1$ (OR), the dotted line that of $h_2$ (AND), and the background is shaded by the whole network's
prediction (blue: $\hat y = 1$, orange: $\hat y = 0$). The region predicted 1 is the band **between** the two lines,
something a single straight line can't draw. On the right is the hidden space $(h_1, h_2)$, with the output neuron's
decision boundary (green); the number next to each point says how many XOR points land there.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-0.5, 1.5, 201), np.linspace(-0.5, 1.5, 201))
grid = np.c_[g1.ravel(), g2.ravel()]


def shade(ax, yhat_grid, alpha=0.13):
    """Shade the region predicted 1 (blue) and the region predicted 0 (orange) on the grid g1, g2."""
    ax.contourf(g1, g2, yhat_grid.reshape(g1.shape), levels=[-0.5, 0.5, 1.5], colors=[ORANGE, BLUE], alpha=alpha)


fig, (a, b) = plt.subplots(1, 2, figsize=(9, 4.4))
shade(a, step(step(grid @ W1 + b1) @ W2 + b2)[:, 0])
for k, ls in [(0, "--"), (1, ":")]:
    a.contour(
        g1, g2, (grid @ W1[:, k] + b1[k]).reshape(g1.shape), levels=[0], colors="black", linewidths=1.8, linestyles=ls
    )
a.plot([], [], "k--", lw=1.8, label="line of h₁ (OR)")
a.plot([], [], "k:", lw=1.8, label="line of h₂ (AND)")
a.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=9)
show_points(a, legend=False)
a.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x₁", ylabel="x₂", title="Input space: two lines")

shade(b, step(z_of(grid, w_h, b_h)))
b.contour(g1, g2, z_of(grid, w_h, b_h).reshape(g1.shape), levels=[0], colors=GREEN, linewidths=2.5)
show_points(b, H_hand, Y)
for h in np.unique(H_hand, axis=0):
    count = int((H_hand == h).all(axis=1).sum())
    b.annotate(f"{count} point{'s' if count > 1 else ''}", h, xytext=(12, 10), textcoords="offset points", fontsize=9)
b.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="h₁ (OR)", ylabel="h₂ (AND)", title="Hidden space: one line is enough")
for ax in (a, b):
    ax.set_aspect("equal")
    ax.set(xticks=[-0.5, 0, 0.5, 1, 1.5], yticks=[-0.5, 0, 0.5, 1, 1.5])
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

This is the main idea of the lesson: **the hidden layer doesn't classify; it changes how the data is represented**,
so that the next layer, still just a perceptron, can do its job. A multilayer perceptron (MLP) is a stack of
transformations like this.

### 🤔 Predict first: remove the activation function
Keep $W_1$, $b_1$ but drop $f$ in the hidden layer: $h = xW_1 + b_1$. Can the perceptron on the new points $h$ solve
XOR?

In [ ]:
H_lin = P @ W1 + b1  # a hidden layer without an activation function
lin_run = perceptron(H_lin, Y, [0.3, -0.2], 0.1, max_updates=1000)
print("the 4 points in the linear hidden space:", "  ".join(f"({num(h[0], 1)}, {num(h[1], 1)})" for h in H_lin))
print(f"perceptron on h: {num(len(lin_run) - 1, 0)} updates, at most {max(s[2] for s in lin_run)}/4 points correct")
assert max(s[2] for s in lin_run) < 4

No. Without $f$, the two columns of $W_1$ here are identical, so $h_2 = h_1 - 1$: the four points lie on **one
straight line**, in the order $(0,0)$, then $(0,1)$ and $(1,0)$ on top of each other, then $(1,1)$; the labels go
0, 1, 1, 0, and no straight line can cut them that way. More generally (Lesson 02):
$(xW_1 + b_1)W_2 + b_2 = x(W_1 W_2) + (b_1 W_2 + b_2)$ is still **one** linear transformation, so the decision
boundary is still a straight line, whatever $W_1$ you choose. It takes a nonlinear activation function to “bend” the
space.

## 6 · Letting the network learn the weights

In section 4 we chose the weights ourselves. Can the network learn them? The perceptron rule doesn't work for the
hidden layer: the hidden layer has no labels, so there is no way to know what the “right” $h$ is. The modern
approach (section 0.5) is:

1. Replace the step function with the **sigmoid** $\sigma(z) = 1/(1 + e^{-z})$ in both layers (Lesson 02), so that
   the output $\hat y \in (0, 1)$ changes smoothly with each weight.
2. Measure the error with a single number, here the mean squared error over the four points,
   $\mathcal L = \tfrac14 \sum_i (\hat y_i - y_i)^2$ (Lesson 04 will discuss which loss functions work better).
3. Repeat: nudge each parameter a little in the direction that makes $\mathcal L$ decrease:
   $\theta \leftarrow \theta - \eta\, \partial \mathcal L / \partial \theta$ (gradient descent, Lesson 05).

To avoid getting ahead of Lesson 06, we compute the derivative $\partial \mathcal L / \partial \theta$ by **trial**:
nudge the parameter $\theta_k$ up and down by $\epsilon = 10^{-4}$ and see how much $\mathcal L$ changes (central
differences). This is slow, but with 9 parameters it doesn't matter, and we run 100 networks (100 initialization
seeds) in parallel: the array `theta` has shape 100 × 9, and each row holds all the parameters of one network. Every
parameter is initialized from the normal distribution $N(0, 1)$; we use $\eta = 4$ and 1,000 steps (Lesson 05
discusses how to choose $\eta$).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def unpack(theta, H):
    """theta (..., 4H + 1) → W₁ (..., 2, H), b₁ (..., H), W₂ (..., H), b₂ (...)."""
    W1 = theta[..., : 2 * H].reshape(*theta.shape[:-1], 2, H)
    return W1, theta[..., 2 * H : 3 * H], theta[..., 3 * H : 4 * H], theta[..., 4 * H]


def mlp(theta, H, X=P):
    """h = σ(xW₁ + b₁), ŷ = σ(hW₂ + b₂) for every row of X and every network in theta."""
    W1, b1, W2, b2 = unpack(theta, H)
    h = sigmoid(X @ W1 + b1[..., None, :])  # (..., number of points, H)
    return sigmoid((h @ W2[..., None])[..., 0] + b2[..., None]), h


def loss(theta, H):
    return ((mlp(theta, H)[0] - Y) ** 2).mean(axis=-1)


def num_grad(theta, H, eps=1e-4):
    """∂L/∂θₖ ≈ [L(θ + εeₖ) − L(θ − εeₖ)] / 2ε, for every k at once."""
    E = eps * np.eye(theta.shape[-1])
    return (loss(theta[..., None, :] + E, H) - loss(theta[..., None, :] - E, H)) / (2 * eps)


def init_theta(H, seeds=range(100), scale=1.0):
    """One row of parameters per seed: every weight and bias drawn from N(0, scale²)."""
    return np.stack([np.random.default_rng(s).normal(0, scale, 4 * H + 1) for s in seeds])


def train_xor(H, seeds=range(100), steps=1000, eta=4.0, scale=1.0, theta=None):
    """Gradient descent for many 2 → H → 1 networks at once. Returns the final parameters, the loss curves, and which
    networks solved XOR."""
    theta = init_theta(H, seeds, scale) if theta is None else theta
    curve = [loss(theta, H)]
    for _ in range(steps):
        theta = theta - eta * num_grad(theta, H)
        curve.append(loss(theta, H))
    solved = ((mlp(theta, H)[0] >= 0.5) == (Y == 1)).all(axis=-1)  # all 4 right when thresholding at ŷ ≥ 0.5
    return theta, np.array(curve), solved


theta2, curve2, solved2 = train_xor(H=2)  # 100 networks 2 → 2 → 1, 9 parameters each, η = 4, 1,000 steps
yhat0, h0 = mlp(theta2[0], 2)
print(
    f"network seed 0: L = {num(curve2[0, 0], 4)} at the start → {num(curve2[-1, 0], 4)} after {num(len(curve2) - 1, 0)} steps"
)
for p, h, yh in zip(P, h0, yhat0):
    print(f"  x = ({p[0]:.0f}, {p[1]:.0f})  →  h = ({num(h[0])}, {num(h[1])})  →  ŷ = {num(yh, 3)}")
assert solved2[0] and curve2[-1, 0] < 0.002

The seed-0 network learns XOR: $\hat y$ is close to 1 at the two points labeled 1 and close to 0 at the two points
labeled 0. But its hidden layer **doesn't look like** the OR/AND solution we set by hand: one hidden neuron turns on
for $(0,1)$, the other for $(1,0)$, and the two points labeled 0 are pushed close to $(0, 0)$. The same problem has
many good hidden representations; gradient descent finds one, not necessarily the one a human would come up with.

The figure below draws this network like the figure in section 5: on the left, the dashed and dotted lines are where
each hidden neuron goes from below to above 0.5, and the background is shaded by $\hat y \ge 0.5$; on the right, the
four points in the hidden space $(h_1, h_2)$ and the output neuron's line $\hat y = 0.5$ (green).

In [ ]:
def show_learned(theta, title):
    """A trained 2 → 2 → 1 network: the input space (left) and the hidden space (right), like the figure in section 5."""
    _, (a, b) = plt.subplots(1, 2, figsize=(9, 4.4))
    yhat_grid, h_grid = mlp(theta, 2, X=grid)
    shade(a, (yhat_grid >= 0.5).astype(int))
    for k, ls in [(0, "--"), (1, ":")]:
        a.contour(g1, g2, h_grid[:, k].reshape(g1.shape), levels=[0.5], colors="black", linewidths=1.8, linestyles=ls)
    show_points(a, legend=False)
    a.set(xlabel="x₁", ylabel="x₂", title=f"{title}: input space")
    _, _, W2_, b2_ = unpack(theta, 2)
    shade(b, step(grid @ W2_ + b2_))  # in the hidden space: ŷ ≥ 0.5 ⇔ hW₂ + b₂ ≥ 0
    b.contour(g1, g2, (grid @ W2_ + b2_).reshape(g1.shape), levels=[0], colors=GREEN, linewidths=2.5)
    show_points(b, mlp(theta, 2)[1], Y)
    b.set(xlabel="h₁", ylabel="h₂", title=f"{title}: learned hidden space")
    for ax in (a, b):
        ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xticks=[-0.5, 0, 0.5, 1, 1.5], yticks=[-0.5, 0, 0.5, 1, 1.5])
        ax.set_aspect("equal")
    plt.tight_layout()
    fmt_axes(plt.gcf())
    plt.show()


show_learned(theta2[0], "Seed 0")

On the right, the two orange points huddle near the origin and the two blue points sit at the other two corners: a
straight line separates them, just as with the hand-set solution.

### Checking section 0.5: how wrong are central differences?
By section 0.5, the derivative that `num_grad` computes is off from the true derivative by about
$\tfrac{\varepsilon^2}{6}\,\partial_k^3 \mathcal L$: the error divided by $\varepsilon^2$ is almost constant, and
halving $\varepsilon$ divides the error by about 4. The cell below takes the 100 starting points of the 2 → 2 → 1
networks above and computes the gradient with three values of $\varepsilon$ much larger than the training's
$10^{-4}$. The reference is the gradient computed with $\varepsilon = 10^{-4}$: by the same formula, its error is
about $(0.025 / 10^{-4})^2$ times smaller. The cell prints the largest deviation (over the 100 networks and their
9 parameters), that deviation divided by $\varepsilon^2$, and by how much the error of each network shrinks every time
$\varepsilon$ is halved.

In [ ]:
theta_init = init_theta(2)  # the 100 starting points of the 2 → 2 → 1 networks above
g_ref = num_grad(theta_init, 2)  # ε = 10⁻⁴ as in training: the reference
fd_err = {}
for eps in [0.1, 0.05, 0.025]:
    fd_err[eps] = np.abs(num_grad(theta_init, 2, eps) - g_ref).max(axis=-1)  # the worst parameter, network by network
    print(
        f"ε = {num(eps, 3)}: largest deviation {num(fd_err[eps].max() * 1e6, 1)}·10⁻⁶, divided by ε²: {num(fd_err[eps].max() / eps**2, 4)}"
    )
for big, small in itertools.pairwise(fd_err):
    ratio = fd_err[big] / fd_err[small]
    print(
        f"ε from {num(big, 3)} down to {num(small, 3)}: each network's error shrinks {num(ratio.min())} to {num(ratio.max())} times"
    )
    assert np.all(np.abs(ratio - 4) < 0.1)

The deviation divided by $\varepsilon^2$ is almost constant, and every time $\varepsilon$ is halved, the error of all
100 networks shrinks 3.96 to 4.04 times, then 3.99 to 4.01 times: the error is proportional to $\varepsilon^2$,
exactly as the Taylor expansion of section 0.5 says. With the training's $\varepsilon = 10^{-4}$,
$\varepsilon^2 = 10^{-8}$, so the error is only about $0.0211 \cdot 10^{-8} \approx 2 \cdot 10^{-10}$: `num_grad`
gives the gradient accurately, just slowly.

### 🤔 Predict first
Two hidden neurons are **enough** to represent XOR (section 4 just did it by hand). So, of 100 randomly initialized
2 → 2 → 1 networks (100 seeds), with the same η and the same 1,000 steps, how many learn XOR?

In [ ]:
lo, hi = curve2[-1].min(), curve2[-1].max()
print(f"solved XOR: {solved2.sum()}/100 seeds;  final L from {num(lo, 4)} to {num(hi, 4)}")
print(f"  the solved networks: final L ≤ {num(curve2[-1, solved2].max(), 4)}")
print(
    f"  the stuck networks: final L from {num(curve2[-1, ~solved2].min(), 4)} to {num(curve2[-1, ~solved2].max(), 4)}"
)
assert abs(solved2.sum() - 77) <= 3, "far from the result when this notebook was written (77/100)"

fig, ax = plt.subplots(figsize=(8, 3.4))
steps_axis = np.arange(curve2.shape[0])
for s in range(100):
    ok = solved2[s]
    ax.plot(steps_axis, curve2[:, s], color=BLUE if ok else ORANGE, ls="-" if ok else "--", lw=0.8, alpha=0.6)
ax.plot([], [], color=BLUE, label=f"solved ({solved2.sum()} seeds)")
ax.plot([], [], color=ORANGE, ls="--", label=f"stuck ({(~solved2).sum()} seeds)")
ax.set(yscale="log", xlabel="gradient descent step", ylabel="loss L (log scale)", xlim=(0, steps_axis[-1]))
ax.set_title("100 networks 2 → 2 → 1 on XOR, one curve per seed")
ax.legend(frameon=False)
fmt_axes(plt.gcf())
plt.show()

Nearly a quarter of the networks fail to learn, even though two hidden neurons are enough. On the plot, the blue
curves (solved) plunge after a few hundred steps; the dashed orange curves (stuck) stop at an almost horizontal level
and go no lower. **Being able to represent something doesn't mean being able to learn it.**

### What does a stuck network look like?
The cell below measures, in each network's hidden space, the smallest distance between two points with **different
labels**, and the gradient magnitude $\lVert \partial \mathcal L / \partial \theta \rVert$ at the end compared with
at initialization. Then it runs the stuck networks for another 4,000 steps.

In [ ]:
_, h_all = mlp(theta2, 2)  # (100 networks, 4 points, 2 hidden neurons)
opposite = [(i, j) for i, j in itertools.combinations(range(4), 2) if Y[i] != Y[j]]
gap = np.min([np.linalg.norm(h_all[:, i] - h_all[:, j], axis=-1) for i, j in opposite], axis=0)
g_end = np.linalg.norm(num_grad(theta2, 2), axis=-1)
g_start = np.linalg.norm(num_grad(init_theta(2), 2), axis=-1)
print("smallest distance between two differently labeled points in the hidden space:")
print(f"  solved networks: {num(gap[solved2].min(), 2)} to {num(gap[solved2].max(), 2)}")
print(f"  stuck networks:  {num(gap[~solved2].min(), 4)} to {num(gap[~solved2].max(), 4)}")
print(
    f"median |gradient|: {num(np.median(g_start), 4)} at the start, {num(np.median(g_end[~solved2]), 4)} in the stuck networks"
)
print(f"  that is, about {num(np.median(g_start) / np.median(g_end[~solved2]), 0)} times smaller")
stuck_seeds = np.flatnonzero(~solved2)
near_eighth = curve2[-1, stuck_seeds] < 0.13
other = stuck_seeds[~near_eighth][0]
print(
    f"{near_eighth.sum()}/{len(stuck_seeds)} stuck networks have L close to 0.125; the other one (seed {other}) has ŷ =",
    end=" ",
)
print("  ".join(num(v, 3) for v in mlp(theta2[other], 2)[0]))
theta_more, _, solved_more = train_xor(2, theta=theta2[stuck_seeds], steps=4000)
print(f"another 4,000 steps: {solved_more.sum()}/{len(stuck_seeds)} stuck networks escape;", end=" ")
norm_before, norm_after = np.linalg.norm(theta2[stuck_seeds], axis=1), np.linalg.norm(theta_more, axis=1)
print(f"median |θ| {num(np.median(norm_before), 1)} → {num(np.median(norm_after), 1)}")
print(f"seed {stuck_seeds[0]}: ŷ =", "  ".join(num(v, 3) for v in mlp(theta2[stuck_seeds[0]], 2)[0]))
assert gap[~solved2].max() < 0.05 < gap[solved2].min()

In [ ]:
show_learned(theta2[stuck_seeds[0]], f"Seed {stuck_seeds[0]}")

The stuck networks have **merged the wrong points**. In section 5, the hidden layer merged two points with the
*same label*, $(0,1)$ and $(1,0)$, and that was exactly the solution. Here the hidden layer merges two points with
*different labels* into almost the same place (right figure: an orange circle and a blue square on top of each
other). The output neuron can't separate two coinciding points, so it answers $\hat y \approx 0.5$ for both: two
errors of $0.5^2$ over four points give $\mathcal L \approx 2 \cdot 0.25 / 4 = 0.125$, exactly the level where the
orange curves stop. (The other stuck network merges *three* points, two orange and one blue, and answers
$\hat y \approx 1/3$ for all three: $\mathcal L \approx 1/6$.) To pull those two points apart, the hidden neurons
would have to leave the sigmoid's saturation region (Lesson 02), where the slope is close to 0: the gradient is more
than a hundred times smaller than at the start, so another 4,000 steps rescue only 1 network.

This is not quite a “local minimum”. For the 2 → 2 → 1 sigmoid network on XOR, Sprinkhuizen-Kuyper & Boers (1998)
proved that every stationary point with finite weights is either a saddle point or a global minimum with zero error.
A stuck network is drifting very slowly across an almost flat region, with weights that keep growing ($|\theta|$
increases in the cell above). For whoever is training it, the outcome is the same: it doesn't learn in a finite
amount of time. (Lesson 05 discusses flat regions and saddle points.)

## 7 · More hidden neurons

The theory of this section is in sections 0.6 and 0.7.

### Wider is easier to train
Rerun exactly the 100-seed experiment with $H$ = 1, 2, 3, 4 hidden neurons (each network has $4H + 1$ parameters):

In [ ]:
width = {}
for H in [1, 2, 3, 4]:
    _, curve_H, solved_H = (theta2, curve2, solved2) if H == 2 else train_xor(H)
    width[H] = int(solved_H.sum())
    print(
        f"H = {H}: {4 * H + 1:>2} parameters, solved {width[H]:>3}/100 seeds, median final L {num(np.median(curve_H[-1]), 4)}"
    )
assert width[1] == 0 and width[4] >= 95

One hidden neuron is never enough (section 0.6): a 1-dimensional hidden layer places the four points on a straight
line, and the labels 0, 1, 1, 0 can't be cut by a single threshold. Two neurons are enough to represent XOR but
often get stuck; add one or two more and almost every seed learns. A network with spare neurons has more ways to
merge correctly, so it is less likely that all of its neurons merge the wrong points at once. This is one reason
real networks are usually larger than the minimum needed to represent the problem.

### More neurons, more lines, more complex regions
Each step-function hidden neuron is a straight line that cuts the plane in two. The output neuron combines those
half-planes. For example: to recognize the points **inside the circle** of radius 1, use $H$ neurons, each one a
tangent line of the circle, then let the output turn on when **all $H$** neurons are on (an AND, like $h_2$ in
section 4). The region it recognizes is a regular polygon with $H$ sides, circumscribed about the circle. The cell
below measures the area where it is wrong (on a grid of points) and compares it with the geometric formula
$H\tan(\pi/H) - \pi$.

In [ ]:
gx, gy = np.meshgrid(np.linspace(-2.1, 2.1, 841), np.linspace(-2.1, 2.1, 841))  # the triangle has vertices at radius 2
plane = np.c_[gx.ravel(), gy.ravel()]
in_disk = (plane**2).sum(axis=1) <= 1
cell_area = (4.2 / 840) ** 2  # area of each grid cell


def polygon_net(H):
    """Hidden layer: H tangent lines (normals pointing toward the center); output: the AND of all H neurons."""
    angle = 2 * np.pi * np.arange(H) / H
    W1 = -np.stack([np.cos(angle), np.sin(angle)])  # 2 × H; neuron k turns on when x·uₖ ≤ 1
    b1 = np.ones(H)
    W2, b2 = np.ones(H), -(H - 0.5)  # on when all H hidden neurons are on
    return lambda X: step(step(X @ W1 + b1) @ W2 + b2)


fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, H in zip(axes, [3, 4, 8, 32]):
    inside = polygon_net(H)(plane)
    err = (inside != in_disk).sum() * cell_area
    exact = H * np.tan(np.pi / H) - np.pi
    print(
        f"H = {H:>2}: error area {num(err, 3)} (formula: {num(exact, 3)}), i.e. {pct(exact / np.pi, 1)} of the circle's area"
    )
    ax.contourf(gx, gy, inside.reshape(gx.shape), levels=[0.5, 1.5], colors=[BLUE], alpha=0.35)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="black", lw=1.2, ls="--"))
    ax.set(xlim=(-2.1, 2.1), ylim=(-2.1, 2.1), xticks=[], yticks=[], title=f"H = {H} hidden neurons")
    ax.set_aspect("equal")
    assert abs(err - exact) < 0.02
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

The blue region is where the network answers “inside”; the dashed line is the true circle. With 3 neurons, the
triangle overshoots by 65.4% of the circle's area; with 32 neurons, only about 0.3% is left. The more neurons, the
closer the polygon hugs the circle. This is the intuition behind the **universal approximation theorem** (Cybenko
1989; Hornik 1991; Leshno et al. 1993; stated precisely in section 0.7): with an activation function that is not a
polynomial, a single hidden layer that is wide enough can approximate any continuous function on a closed and
bounded domain to arbitrarily small error.

The theorem only says that such a network **exists**. It doesn't say how many neurons are needed (for the circle
here, the error area shrinks like $\pi^3 / 3H^2$), and it doesn't say whether gradient descent will find those
weights: section 6 just showed that even when the network is large enough, learning can still get stuck. Here we set
the weights by hand; the theorem gives no way to find them.

### Deep or wide?
Adding neurons to **one** hidden layer makes the network wider. Adding **layers** makes it deeper. With the same
number of neurons, which way can draw more complex shapes? A one-dimensional example (Telgarsky, 2015; proved in
section 0.7) gives a clear answer. Take the “roof” function on $[0, 1]$:

$$g(x) = 2\,\mathrm{ReLU}(x) - 4\,\mathrm{ReLU}(x - 0.5),$$

that is, one hidden layer with **2** ReLU neurons: $g$ goes up from 0 to 1 and back down to 0. Stack $k$ such layers
on top of each other ($g$ of $g$ of $g$…) and each layer doubles the number of linear pieces. The cell below counts
the linear pieces of the function the network computes.

In [ ]:
def relu(z):
    return np.maximum(0, z)


def g(x):
    return 2 * relu(x) - 4 * relu(x - 0.5)  # one hidden layer, 2 ReLU neurons


x_line = np.linspace(0, 1, 2**12 + 1)  # the points k/4096: the arithmetic on them is exact


def n_pieces(y):
    """Number of linear pieces of a piecewise linear function, counted at the places where the slope changes value."""
    slope = np.diff(y) / np.diff(x_line)
    return 1 + int((np.diff(slope) != 0).sum())


fig, axes = plt.subplots(1, 3, figsize=(10, 2.6), sharey=True)
y_line = x_line
for k in range(1, 7):
    y_line = g(y_line)  # add one layer
    print(
        f"{k} layer{'s' if k > 1 else ' '} × 2 neurons = {2 * k:>2} neurons: {n_pieces(y_line):>2} linear pieces;  "
        f"1 layer needs ≥ {2**k - 1:>2} neuron{'s' if k > 1 else ''}"
    )
    assert n_pieces(y_line) == 2**k
    if k in (1, 2, 4):
        ax = axes[[1, 2, 4].index(k)]
        ax.plot(x_line, y_line, color=BLUE, lw=1.6)
        ax.set(xlabel="x", title=f"{k} layer{'s' if k > 1 else ''} ({2 * k} neurons): {2**k} pieces")
axes[0].set_ylabel("output")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

With $k$ layers of 2 neurons each, the function has $2^k$ linear pieces. A single hidden layer with $m$ ReLU neurons,
on the other hand, gives at most $m + 1$ pieces along a line, because each neuron adds only one kink. To get 64 pieces
like 6 layers × 2 neurons (12 neurons), a one-layer network needs at least 63 neurons. **Depth multiplies the number
of regions; width only adds to it.**
Telgarsky proved that there are functions a deep network computes exactly, while every shallow network needs a number
of neurons that grows exponentially to approximate them. That doesn't mean deeper is always better (deep networks are
harder to train: Chapter 3 of the course), but it explains why practical networks have many layers instead of one
very wide layer.

### Checking section 0.7: how many pieces does one layer of $m$ ReLU neurons give?
By section 0.7, each neuron of a hidden layer adds exactly one kink: a layer of $m$ neurons has exactly $m + 1$
pieces on $[0, 1]$ when the kinks are distinct and lie in $(0, 1)$, and never more. The cell below builds 100 random
one-layer networks for each $m$: the kinks are distinct, drawn at random from the grid points $j/4096$, and the kink
directions and output weights are integers, so that the arithmetic stays exact as in the cell above. It counts the
pieces with the same function `n_pieces` and prints, for each $m$, the piece counts it found, next to $m + 1$.

In [ ]:
rng_pl = np.random.default_rng(0)
one_layer = {}
for m in [1, 2, 3, 12, 63]:
    counts = set()
    for _ in range(100):
        kinks = rng_pl.choice(np.arange(1, 4096), size=m, replace=False) / 4096  # m distinct kinks in (0, 1)
        signs = rng_pl.choice([-1, 1], size=m)  # neuron j is ReLU(x − kinks[j]) or ReLU(kinks[j] − x)
        a_out = rng_pl.choice([-3, -2, -1, 1, 2, 3], size=m)  # output weights: nonzero integers
        counts.add(n_pieces(relu(signs * (x_line[:, None] - kinks)) @ a_out + rng_pl.integers(-3, 4)))
    one_layer[m] = sorted(counts)
    print(
        f"one layer of m = {m:>2}: 100 random networks give {', '.join(map(str, one_layer[m]))} pieces  (m + 1 = {m + 1})"
    )
print(f"same 12 neurons: 1 layer gives {one_layer[12][0]} pieces, 6 layers × 2 neurons (cell above) {n_pieces(y_line)}")
assert all(one_layer[m] == [m + 1] for m in one_layer)
assert one_layer[12] == [13], "differs from the lesson page (12 neurons in one layer: at most 13 pieces): check again"

For every $m$, all 100 networks have exactly $m + 1$ pieces: each neuron adds exactly one kink, no more. Kinks that
coincide or fall outside $(0, 1)$ only lower the count (section 0.7), so $m + 1$ is an upper bound. With the same
12 neurons, one layer gives 13 pieces and 6 layers give 64; a single layer needs 63 neurons to reach 64 pieces.

## 8 · A real MLP: the MNIST network 784 → 128 → 64 → 10

The network that accompanies us through the next lessons (MLSysBook's MNIST example) is an MLP with two hidden
layers, with the same row-vector convention (section 0.8):

$$h_1 = \mathrm{ReLU}(x W_1 + b_1), \qquad h_2 = \mathrm{ReLU}(h_1 W_2 + b_2), \qquad z = h_2 W_3 + b_3$$

$x$ is an image of 784 pixels; $z$ consists of 10 scores, one per digit, and the network picks the digit with the
highest score (Lesson 04 turns those 10 scores into probabilities with softmax). Counting MACs and parameters:

In [ ]:
SIZES = [784, 128, 64, 10]
for n_in, n_out in itertools.pairwise(SIZES):
    print(
        f"layer {n_in:>3} → {n_out:>3}: W of size {n_in}×{n_out}, {num(n_in * n_out, 0):>7} MACs, {num(n_in * n_out + n_out, 0):>7} parameters"
    )
macs = sum(n_in * n_out for n_in, n_out in itertools.pairwise(SIZES))
n_par = sum(n_in * n_out + n_out for n_in, n_out in itertools.pairwise(SIZES))
print(f"total: {num(macs, 0)} MACs per image, {num(n_par, 0)} parameters")
print(f"section 6's num_grad would need {num(2 * n_par, 0)} forward passes per training step")
assert macs == 109184 and n_par == 109386

109,184 MACs, matching the lesson page. The cell below trains the network exactly as the course's
`reel/mnist_mlp/train.py` does (same seed, so the same numbers as the demos of later lessons): He initialization
(Lesson 09), a mini-batch of 64 images per step, $\eta = 0.1$, 5 epochs. Two things differ from section 6:

- The output uses softmax and cross-entropy instead of a sigmoid and the squared error (Lesson 04 explains why).
- The gradient isn't computed by trial: as the cell above just printed, `num_grad` needs hundreds of thousands of
  forward passes for *each* step. The three lines marked “backpropagation” compute exactly that gradient at a cost
  of about one forward pass. Lesson 06 explains them; here, just use them as a black box.

In [ ]:
def forward(params, X):
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = relu(X @ W1 + b1)
    h2 = relu(h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


rng = np.random.default_rng(0)
params = [
    [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
]
params0 = [[W.copy(), b.copy()] for W, b in params]  # keep the network as it was before training
for epoch in range(1, 6):
    order = rng.permutation(len(Xtr))
    for i in range(0, len(order), 64):
        idx = order[i : i + 64]
        x, y = Xtr[idx], ytr[idx]
        h1, h2, z = forward(params, x)
        dz = softmax(z)
        dz[np.arange(len(y)), y] -= 1
        dz /= len(y)
        (W1_, _), (W2_, _), (W3_, _) = params
        dh2 = (dz @ W3_.T) * (h2 > 0)  # backpropagation (Lesson 06)
        dh1 = (dh2 @ W2_.T) * (h1 > 0)  # backpropagation
        grads = [(x.T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]  # backpropagation
        for layer, (gW, gb) in zip(params, grads):
            layer[0] -= 0.1 * gW
            layer[1] -= 0.1 * gb
    acc = (forward(params, Xte)[2].argmax(axis=1) == yte).mean()
    print(f"epoch {epoch}: {pct(acc)} correct on {num(len(yte), 0)} test images")
assert abs(acc - 0.9727) < 0.0005, "differs from the demos of later lessons (97.27%): check the data"

97.27% after 5 epochs, exactly the number in the demos of later lessons; the 10-class perceptron of Lesson 01 stopped
at 88.70% (the next cell reruns it).

### What has the MNIST network's hidden layer learned?
With XOR we could see the hidden space with our own eyes. With MNIST, $h_1$ has 128 dimensions, so we measure it
instead: we let **the 10-class perceptron of Lesson 01** (one epoch, $\eta = 0.1$, seed 0) learn on different
features in place of the pixels. If a hidden layer makes the data “more separable”, a simple perceptron placed on top
of it will get more right.

### 🤔 Predict first
On the pixels, the perceptron gets 88.70% right (Lesson 01). On $h_1$ of the **untrained network** (128 ReLU neurons,
random weights from initialization), does it get more right or fewer? And on $h_1$ and $h_2$ of the trained network?

In [ ]:
def perceptron10(F, y, F_test, y_test, seed=0, eta=0.1):
    """The 10-class perceptron of Lesson 01: on a mistake, add ηx to the column of the correct digit and subtract ηx
    from the column of the digit it was mistaken for."""
    W, b, fixes = np.zeros((F.shape[1], 10)), np.zeros(10), 0
    for i in np.random.default_rng(seed).permutation(len(y)):
        k = int(np.argmax(F[i] @ W + b))
        if k != y[i]:
            W[:, y[i]] += eta * F[i]
            b[y[i]] += eta
            W[:, k] -= eta * F[i]
            b[k] -= eta
            fixes += 1
    return fixes, ((F_test @ W + b).argmax(axis=1) == y_test).mean()


h1_tr, h2_tr, _ = forward(params, Xtr)
h1_te, h2_te, _ = forward(params, Xte)
features = {
    "pixels x (784 dims)": (Xtr, Xte),
    "h₁ of the untrained network (128)": (forward(params0, Xtr)[0], forward(params0, Xte)[0]),
    "h₁ of the trained network (128)": (h1_tr, h1_te),
    "h₂ of the trained network (64)": (h2_tr, h2_te),
}
probe = {}
for name, (F, F_test) in features.items():
    fixes, probe[name] = perceptron10(F, ytr, F_test, yte)
    print(f"perceptron on {name:<33}: {num(fixes, 0):>6} updates, {pct(probe[name])} correct")
assert abs(probe["pixels x (784 dims)"] - 0.887) < 0.002, "differs from Lesson 01 (88.7%)"
assert (
    probe["h₁ of the untrained network (128)"] < probe["pixels x (784 dims)"] < probe["h₂ of the trained network (64)"]
)

The random hidden layer is even **worse** than the pixels (85.78% against 88.70%): by itself, it doesn't make the
data easier to separate. After training, a perceptron placed on $h_1$ or $h_2$ gets 96.16–96.45% right, close to the
whole network's 97.27%, even though it learns only with Lesson 01's rule, in a single epoch. The hard part of the work
is already done in the hidden layers: they turn the 10 classes of digits into clusters of points that are almost
separable by hyperplanes, just as XOR's hidden layer turned four points into three separable ones.

Lesson 01's hard pair of digits, 4 and 9, shows this more clearly. The cell below reruns exactly the two-class
perceptron of Lesson 01 on the pixels and on $h_2$, then projects the test images onto **the two directions along
which the data spreads the most** (principal component analysis, PCA; Lesson 19 uses it again) so that we can see
them.

In [ ]:
def perceptron_pass(X, y, eta=0.1, seed=0):
    """The two-class perceptron of Lesson 01: w = 0, one epoch in random order, update only on a mistake."""
    w, b, fixes = np.zeros(X.shape[1]), 0.0, 0
    for i in np.random.default_rng(seed).permutation(len(y)):
        y_hat = int(X[i] @ w + b >= 0)
        if y_hat != y[i]:
            w, b, fixes = w + eta * (y[i] - y_hat) * X[i], b + eta * (y[i] - y_hat), fixes + 1
    return w, b, fixes


keep_tr, keep_te = np.isin(ytr, [4, 9]), np.isin(yte, [4, 9])
y49, y49_te = (ytr[keep_tr] == 4).astype(int), (yte[keep_te] == 4).astype(int)  # y = 1 means "4"
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (name, F, F_te) in zip(
    axes, [("pixels x", Xtr[keep_tr], Xte[keep_te]), ("h₂ of the trained network", h2_tr[keep_tr], h2_te[keep_te])]
):
    w, b, fixes = perceptron_pass(F, y49)
    n_ok = int(((F_te @ w + b >= 0) == y49_te).sum())
    print(
        f"4 or 9 on {name}: {fixes} updates, {num(n_ok, 0)}/{num(len(y49_te), 0)} = {pct(n_ok / len(y49_te))} correct on the test set"
    )
    if name == "pixels x":
        assert (fixes, n_ok) == (798, 1889), "differs from Lesson 01"
    C = F_te - F_te.mean(axis=0)
    proj = C @ np.linalg.svd(C, full_matrices=False)[2][:2].T  # coordinates on the two principal components
    for lab, color, marker, txt in [(1, ORANGE, "o", "4s"), (0, BLUE, "s", "9s")]:
        sel = y49_te == lab
        ax.scatter(*proj[sel].T, s=7, color=color, marker=marker, alpha=0.45, lw=0, label=txt)
    ax.set(xlabel="principal component 1", ylabel="principal component 2", xticks=[], yticks=[], title=name)
    ax.legend(frameon=False, markerscale=2.5, fontsize=9)
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

Each dot is one test image (orange circle: a 4, blue square: a 9). On the pixels (left), the two clouds overlap; in
the hidden space $h_2$ (right), they lie almost on two separate sides. With the same perceptron, the number of
updates drops from 798 to less than a third (218), and the accuracy rises from 94.88% to 98.49%. Two dimensions are
only a slice of the 64, but the meaning is the same as with XOR: the hidden layer has learned a **representation** on
which the problem is almost linear.

## 9 · Try changing things

The cells below rerun the same functions with other settings. You can change the values and run them again.

### The initialization scale
Section 6 initializes every parameter from $N(0, 1)$. Change that standard deviation (`scale`) for 100 2 → 2 → 1
networks:

In [ ]:
for scale in [0.1, 0.5, 1.0, 2.0, 4.0]:
    _, curve_s, solved_s = train_xor(2, scale=scale)
    print(
        f"initialization N(0, {num(scale, 1)}²): solved {solved_s.sum():>2}/100 seeds, median final L {num(np.median(curve_s[-1]), 4)}"
    )

Initialization too small: every $z$ is close to 0, so every hidden neuron outputs nearly 0.5 for all four points, the
gradient is tiny, and 1,000 steps aren't enough to get started (median L 0.2500, exactly the level of $\hat y = 0.5$
for every point). Initialization too large: the sigmoids saturate from the start, and many networks merge the wrong
points before they get a chance to learn. Choosing the initialization scale is a whole topic of its own
(He initialization, Lesson 09).

### A random hidden layer, like the Mark I machine
Rosenblatt's Mark I Perceptron (1958–1960) actually **did** have a hidden layer: 400 photocells wired at random,
through a plugboard, to 512 “association units”. Those wires were fixed; only the last layer learned. The cell below
redoes that idea on MNIST: a hidden layer of 512 ReLU neurons with random weights that are never trained, then a
10-class perceptron on top, 3 seeds.

In [ ]:
mark1 = []
for seed in range(3):
    R = np.random.default_rng(seed).normal(0, np.sqrt(2 / 784), (784, 512))  # a random, fixed hidden layer
    fixes, acc_r = perceptron10(relu(Xtr @ R), ytr, relu(Xte @ R), yte)
    mark1.append(acc_r)
    print(f"seed {seed}: random hidden layer of 512 neurons → {num(fixes, 0)} updates, {pct(acc_r)} correct")
print(f"range across seeds: {pct(min(mark1))} – {pct(max(mark1))}  (pixels: {pct(probe['pixels x (784 dims)'])})")

With 512 neurons, the random hidden layer helps a little compared with the pixels (89.27–90.98% over 3 seeds,
against 88.70%), but it is still far from the **trained** hidden layer of 128 neurons (96.45%). This is exactly the
limit that Minsky & Papert analyzed: one learnable layer placed on top of fixed features. Try increasing the number
of random neurons (`512`) to see how many it takes to catch up.

## 10 · Limitations, and the bridge to Lesson 04

This lesson relied on two “stopgaps”:

- **Output and loss.** For XOR, a sigmoid and the squared error are enough. The MNIST network has 10 outputs; to read
  those 10 scores as probabilities and to measure *how* wrong the network is on each image, we need softmax and
  cross-entropy, which the training cell in section 8 used without explaining them. That is Lesson 04.
- **Gradients.** Section 6's “nudge each parameter and see” approach needs 218,772 forward passes of the MNIST
  network per step. Lesson 05 (walking downhill) and Lesson 06 (backpropagation) replace it with a computation that
  is exact and cheap.

## Summary
- No straight line separates XOR (four contradictory inequalities); the perceptron rule falls into a 4-step cycle
  from update 11 on and never gets more than 3/4.
- A hidden layer $h = f(xW_1 + b_1)$ is two (or more) perceptrons side by side. The OR/AND solution merges $(0,1)$
  and $(1,0)$ into one point, and in the hidden space the perceptron of Lesson 01 solves XOR after 2 updates.
- Without a nonlinear $f$, the hidden layer is useless: two linear layers still amount to one straight line.
- Section 6 computes the gradient by central differences; their error is proportional to $\varepsilon^2$ (halving
  $\varepsilon$ divides the error by about 4), so with $\varepsilon = 10^{-4}$ they give the gradient accurately.
- A 2 → 2 → 1 network is enough to represent XOR, but only 77/100 seeds learn it; the rest merge two differently
  labeled points and get stuck at $\mathcal L \approx 0.125$. With 4 hidden neurons, 100/100.
- More neurons draw more complex regions (a 32-sided polygon hugs the circle: universal approximation); more layers
  multiply the number of regions ($2^k$ pieces with $2k$ neurons, against the $2^k - 1$ neurons needed in a single
  layer, because a layer of $m$ neurons gives at most $m + 1$ pieces: every random one-layer network of the check
  gives exactly $m + 1$).
- The MNIST MLP 784 → 128 → 64 → 10: 109,184 MACs, 97.27% after 5 epochs. The perceptron of Lesson 01 placed on the
  trained hidden layer gets 96.45% right, on the pixels 88.70%, on the random hidden layer 85.78%.

## Review questions
1. Write the four inequalities a perceptron must satisfy to solve XOR, and show why they contradict each other.
2. In the hand-set solution of section 4, what does the hidden layer do with the two points $(0,1)$ and $(1,0)$? Why
   is that enough for a straight line to separate the data?
3. If you remove the activation function from the hidden layer, can the 2 → 2 → 1 network still solve XOR, whatever
   $W_1$ you choose? Why?
4. Two hidden neurons are enough to represent XOR. Why do 23 of the 100 networks still fail to learn it, and why do
   more hidden neurons help?
5. At most how many linear pieces can a hidden layer of $m$ ReLU neurons produce along a line? And 6 layers of
   2 neurons each?
6. What does the universal approximation theorem guarantee, and what does it **not** guarantee?

<details><summary>Hints and answers</summary>

1. $b < 0$; $w_2 + b \ge 0$; $w_1 + b \ge 0$; $w_1 + w_2 + b < 0$. Adding the two middle inequalities gives
   $w_1 + w_2 + b \ge -b > 0$, which contradicts the last one.
2. It merges both into the same point $h = (1, 0)$. The four points become three: $(0,0)$ and $(1,1)$ with label 0,
   $(1,0)$ with label 1, and a straight line separates the middle point from the other two.
3. No. $(xW_1 + b_1)W_2 + b_2 = x(W_1W_2) + (b_1W_2 + b_2)$ is still a linear transformation of $x$, so the decision
   boundary is still a straight line.
4. They merge two differently labeled points into almost the same place; the sigmoid saturates there, so the gradient
   is more than a hundred times smaller and the network barely moves from $\mathcal L \approx 0.125$. With more
   neurons, there are more ways to merge correctly, and it is less likely that every neuron merges the wrong points
   at once.
5. $m + 1$ pieces: each neuron adds exactly one kink, so there are at most $m$ kinks (section 0.7; the check at the
   end of section 7 counts exactly $m + 1$). 6 layers × 2 neurons: $2^6 = 64$ pieces; a single layer needs at least
   63 neurons.
6. A single hidden layer that is wide enough, with a non-polynomial activation function, can approximate any
   continuous function on a closed and bounded domain. The theorem doesn't say how many neurons are needed, and
   doesn't say that gradient descent will find those weights.
</details>

## Further reading
- M. Minsky & S. Papert (1969), [Perceptrons](https://doi.org/10.7551/mitpress/11301.001.0001);
  a summary of the book and the debate around it: [Wikipedia](https://en.wikipedia.org/wiki/Perceptrons_(book)).
- W. McCulloch & W. Pitts (1943), [A logical calculus of the ideas immanent in nervous activity](https://doi.org/10.1007/BF02478259):
  networks of threshold neurons can compute any logical expression.
- H. D. Block & S. A. Levin (1970), [On the boundedness of an iterative procedure for solving a system of linear
  inequalities](https://doi.org/10.1090/S0002-9939-1970-0265383-5): the perceptron cycling theorem.
- D. Rumelhart, G. Hinton & R. Williams (1986), [Learning representations by back-propagating
  errors](https://doi.org/10.1038/323533a0).
- G. Cybenko (1989), [Approximation by superpositions of a sigmoidal function](https://doi.org/10.1007/BF02551274);
  K. Hornik (1991), [Approximation capabilities of multilayer feedforward networks](https://doi.org/10.1016/0893-6080(91)90009-T);
  M. Leshno et al. (1993), [non-polynomial activation functions](https://doi.org/10.1016/S0893-6080(05)80131-5).
- I. Sprinkhuizen-Kuyper & E. Boers (1998), [The error surface of the 2-2-1 XOR network](https://doi.org/10.1016/S0893-6080(98)00014-8):
  no local minima with finite weights.
- M. Telgarsky (2015), [Representation benefits of deep feedforward networks](https://arxiv.org/abs/1509.08101): depth
  multiplies the number of regions.
- Goodfellow, Bengio & Courville, [Deep Learning, section 6.1](https://www.deeplearningbook.org/contents/mlp.html):
  XOR with a ReLU hidden layer. M. Nielsen, [Chapter 4](http://neuralnetworksanddeeplearning.com/chap4.html): a visual
  proof of the universal approximation theorem.
- [TensorFlow Playground](https://playground.tensorflow.org/): change the number of neurons and layers, and watch the
  decision region change.
- The Mark I machine: [Wikipedia, Perceptron](https://en.wikipedia.org/wiki/Perceptron). The MNIST example: MLSysBook,
  [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html).

**Next lesson:** [Lesson 04 · Softmax & loss functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/):
turning 10 scores into probabilities, and measuring how wrong the network is.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")